In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import scanpy as sc
import pandas as pd
import doubletdetection

# ignore warnings
import warnings; warnings.simplefilter("ignore")

Path to Good Outcome sample set - 12 samples

In [ ]:
path = f'{ACC_DATA_ROOT}/good_outcome'
path1 = f'{ACC_DATA_ROOT}/poor_outcome'
sample_files = {
    'P05':f'{path}/P05.h5',
    'P03':f'{path}/P03.h5',
    'P06':f'{path}/P06.h5',
    'P01':f'{path}/P01.h5',
    'P08':f'{path}/P08.h5',
    'P09':f'{path}/P09.h5',
    'P02':f'{path}/P02.h5',
    'P10':f'{path}/P10.h5',
    'P11':f'{path}/P11.h5',
    'P12':f'{path}/P12.h5',
    'P04':f'{path}/P04.h5',
    'P13':f'{path}/P13.h5',
    'P19':f'{path1}/P19.h5',
    'poor_P07':f'{path1}/poor_P07.h5',
    'P20':f'{path1}/P20.h5',
    'P22':f'{path1}/P22.h5',
    'P17':f'{path1}/P17.h5',
    'P23':f'{path1}/P23.h5',
    'P15':f'{path1}/P15.h5',
    'P24':f'{path1}/P24.h5',
    'P18':f'{path1}/P18.h5',
    'P21':f'{path1}/P21.h5',
    'P16':f'{path1}/P16.h5',
    'P25':f'{path1}/P25.h5',
}

Path to Poor Outcome Sample set - 12 samples

In [ ]:
adatas = []
for samp, h5file in sample_files.items():
    ad = sc.read_10x_h5(h5file)
    ad.var_names_make_unique()          # ensure unique gene names
    ad.obs['batch'] = samp             # annotate batch/sample ID
    adatas.append(ad)

In [ ]:
# # initializing parameters for finding doublets.
# if initiated outside the loop then the model trained on first sample can bleed into the predictions of
# second sample.
# clf = doubletdetection.BoostClassifier(
#     n_iters=10,
#     clustering_algorithm="louvain",
#     standard_scaling=True,
#     pseudocount=0.1,
#     n_jobs=-1)

In [ ]:
# # simple pre-processing
# for ad in adatas:
#     sc.pp.filter_cells(ad, min_genes=200)
#     sc.pp.filter_genes(ad, min_cells=3)
#     # e.g., compute percent mito if needed:
#     ad.var['mt'] = ad.var_names.str.startswith('MT-')
#     sc.pp.calculate_qc_metrics(ad, qc_vars=['mt'], inplace=True)
#     ad = ad[ad.obs.pct_counts_mt < 10, :]
#     doublets = clf.fit(ad.X).predict(p_thresh=1e-16, voter_thresh=0.5)
#     doublet_score = clf.doublet_score()
#     ad.obs["doublet"] = doublets
#     ad.obs["doublet_score"] = doublet_score
#     ad.uns['doublets_removed'] = ad.obs.doublet.sum()
#     ad = ad[ad.obs.doublet == 0]

In [ ]:
adatas

In [ ]:
cleaned = []

for ad in adatas:
    # ---------- basic QC ----------
    sc.pp.filter_cells(ad, min_genes=200, inplace=True)
    sc.pp.filter_genes(ad, min_cells=3,  inplace=True)

    ad.var["mt"] = ad.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(ad, qc_vars=["mt"], inplace=True)
    ad = ad[ad.obs.pct_counts_mt < 10, :].copy()

    # ---------- doublet detection ----------
    clf = doubletdetection.BoostClassifier(
        n_iters=50,
        clustering_algorithm="louvain",
        standard_scaling=True,
        pseudocount=0.1,
        n_jobs=8          # adjust
    )
    doublets = clf.fit(ad.X).predict(p_thresh=1e-16, voter_thresh=0.5)
    ad.obs["doublet"]        = doublets
    ad.obs["doublet_score"]  = clf.doublet_score()
    ad.uns["doublets_removed"] = int(doublets.sum())

    # ad = ad[ad.obs.doublet == 0, :].copy()        <- dont subset just have the doublet information as its a cancer sample.

    # ---------- prepare for downstream integration ----------
    ad.layers["counts"] = ad.X.copy()
    # sc.pp.normalize_total(ad, target_sum=1e4)
    # sc.pp.log1p(ad)
    # sc.pp.highly_variable_genes(
    #     ad, flavor="seurat_v3", n_top_genes=3000, batch_key="batch"
    #)

    cleaned.append(ad)

adatas = cleaned

In [ ]:
adatas

In [ ]:
adata = sc.concat(
    adatas, 
    keys=list(sample_files.keys()),   
    label='batch', 
    join='outer'                      
)

In [ ]:
adata


In [ ]:
list(set(adata.obs['batch']))

In [ ]:
site = {
'P04':'Parotid',
'P06':'NA',
'P02':'Parotid',
'P12':'Tongue',
'P05':'NA',
'P01':'Tongue',
'P08':'Larynx',
'P13':"Parotid",
'P09':'BOT',
'P10':'Larynx',
'P11':'BOT',
'P03':'NA',
'P15':'BaseofTongue',
'P16':'Maxilla',
'P17':'Submandibular',
'P18':'Maxilla',
'P19':'NA',
'poor_P07':'NA',
'P20':'BaseofTongue',
'P21':'Maxilla',
'P22':'FloorofMouth',
'P23':'Mandible',
'P24':'Maxillarysinus',
'P25':'Maxilla'
}

In [ ]:
adata.obs['site'] = adata.obs['batch'].map(site)

In [ ]:
adata.obs

In [ ]:
# to make sure the raw count information is being concatenated which is present in layers.
assert "counts" in adata.layers and adata.layers["counts"].shape == adata.X.shape

In [ ]:
adata

In [ ]:
# write out concatenated file.
# adata.write_h5ad(f"{ACC_DATA_ROOT}/ouptuts/concatenated_outputs/raw_merged_counts_all.h5ad", compression='lzf')

In [ ]:
adata.obs